# Olist E-Commerce Customer Intelligence

## Day 2 — Data Cleaning

### Objective
Turn the raw Olist tables into reliable, analysis-ready datasets by resolving the
data-quality issues identified in `1_Data_Understanding.ipynb`.

### Principles
- Raw files in `Data/Raw/` are never modified. All cleaning happens in code.
- Every cleaning decision is documented with its reason.
- Issues are handled at the correct grain (order, item, payment, review, customer).
- Records are flagged rather than deleted where they may still be useful for analysis.
- Cleaned tables are saved to `Data/Processed/`.

### Issues to Address (from Day 1)
1. Date columns stored as text.
2. Misspelled product column names.
3. Order status scope for revenue and customer analysis.
4. Date anomalies in delivered and canceled orders.
5. Repeated review_ids and orders with multiple reviews.
6. Duplicate geolocation records per ZIP prefix.
7. Products with missing category and untranslated categories.
8. Multiple payment rows per order and `not_defined` payment types.
9. Derived fields needed for analysis.

In [ ]:
import os
import pandas as pd

In [ ]:
RAW_PATH = "../Data/Raw/"
PROCESSED_PATH = "../Data/Processed/"
os.makedirs(PROCESSED_PATH, exist_ok=True)

files = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

data = {}
for name, file in files.items():
    data[name] = pd.read_csv(RAW_PATH + file)

print("All datasets loaded")

## 1. Dates

In [ ]:
# dates came in as text, converting so I can calculate delivery times and monthly trends
date_columns = {
    "orders": ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
               "order_delivered_customer_date", "order_estimated_delivery_date"],
    "order_items": ["shipping_limit_date"],
    "reviews": ["review_creation_date", "review_answer_timestamp"],
}

for table, cols in date_columns.items():
    for col in cols:
        data[table][col] = pd.to_datetime(data[table][col])

data["orders"].dtypes

## 2. ZIP codes

In [ ]:
# pandas read ZIP prefixes as numbers so 01037 became 1037
# putting the zeros back so they match across customers, sellers and geolocation
data["customers"]["customer_zip_code_prefix"] = data["customers"]["customer_zip_code_prefix"].astype(str).str.zfill(5)
data["sellers"]["seller_zip_code_prefix"] = data["sellers"]["seller_zip_code_prefix"].astype(str).str.zfill(5)
data["geolocation"]["geolocation_zip_code_prefix"] = data["geolocation"]["geolocation_zip_code_prefix"].astype(str).str.zfill(5)

data["customers"].head(3)

## 3. Products

In [ ]:
# 'lenght' typo is in the source file
data["products"] = data["products"].rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length",
})

# 610 products have no category - calling them 'unknown' instead of dropping,
# otherwise their sales would disappear from the totals
data["products"]["product_category_name"] = data["products"]["product_category_name"].fillna("unknown")

# leaving the other missing values (photos, dimensions) as they are, not guessing them
data["products"].isna().sum()

In [ ]:
# which categories aren't in the translation table?
untranslated = set(data["products"]["product_category_name"]) - set(data["category_translation"]["product_category_name"])
untranslated

In [ ]:
# pc_gamer and portateis_cozinha... are missing from the translation table, plus my 'unknown' label
extra = pd.DataFrame({
    "product_category_name": ["pc_gamer", "portateis_cozinha_e_preparadores_de_alimentos", "unknown"],
    "product_category_name_english": ["pc_gamer", "portable_kitchen_food_processors", "unknown"],
})
translation = pd.concat([data["category_translation"], extra], ignore_index=True)

data["products"] = data["products"].merge(translation, on="product_category_name", how="left")

# should be 0 now
data["products"]["product_category_name_english"].isna().sum()

## 4. Orders

In [ ]:
orders = data["orders"]

# flagging orders with dates that don't make sense (not deleting them)
orders["flag_delivered_no_date"] = (orders["order_status"] == "delivered") & orders["order_delivered_customer_date"].isna()
orders["flag_canceled_with_delivery"] = (orders["order_status"] == "canceled") & orders["order_delivered_customer_date"].notna()
orders["flag_delivered_before_purchase"] = orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]

# result: 8 delivered with no date, 6 canceled with a delivery date, 0 delivered before purchase
orders[["flag_delivered_no_date", "flag_canceled_with_delivery", "flag_delivered_before_purchase"]].sum()

In [ ]:
# orders that count for revenue/customer analysis: delivered, has a delivery date, has items
# everything else stays in the table with in_analysis_scope = False
# defining it once here so Python, SQL and Power BI all use the same orders
orders["has_items"] = orders["order_id"].isin(data["order_items"]["order_id"])

orders["in_analysis_scope"] = (
    (orders["order_status"] == "delivered")
    & orders["order_delivered_customer_date"].notna()
    & orders["has_items"]
)

# 96,470 in scope = 96,478 delivered minus the 8 without a date
orders["in_analysis_scope"].value_counts()

In [ ]:
# customer_id changes with every order, customer_unique_id is the actual customer (needed for RFM)
orders = orders.merge(data["customers"][["customer_id", "customer_unique_id"]], on="customer_id", how="left")

orders["purchase_month"] = orders["order_purchase_timestamp"].dt.strftime("%Y-%m")
orders["delivery_days"] = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.days

# estimate has no time part so comparing by day. only meaningful for delivered orders
orders["is_late"] = orders["order_delivered_customer_date"].dt.normalize() > orders["order_estimated_delivery_date"]

data["orders"] = orders
orders.head(3)

## 5. Revenue

In [ ]:
# revenue = price + freight, per item so it can be split by product, category and seller later
# payment values are only used for payment method analysis
items = data["order_items"]
items["item_revenue"] = items["price"] + items["freight_value"]

items.head(3)

## 6. Payments

In [ ]:
payments = data["payments"]

# all 3 not_defined payments are 0, so they don't affect totals - leaving them in
payments[payments["payment_type"] == "not_defined"]

In [ ]:
# some orders are paid in parts - adding them up to get one row per order
# joining the raw payments table would repeat those orders
payments_by_order = payments.groupby("order_id").agg({
    "payment_value": "sum",
    "payment_installments": "max",
}).reset_index()

payments_by_order.head(3)

## 7. Reviews

In [ ]:
# some orders have more than one review, keeping the latest so there's one per order
reviews = data["reviews"].sort_values("review_answer_timestamp", ascending=False)
reviews_by_order = reviews.drop_duplicates("order_id")

print("before:", len(reviews))
print("after:", len(reviews_by_order))

## 8. Geolocation

In [ ]:
geo = data["geolocation"]

# dropping points outside Brazil (roughly its borders)
geo = geo[geo["geolocation_lat"].between(-34, 6)]
geo = geo[geo["geolocation_lng"].between(-74, -34)]

# 1M+ rows but only ~19k ZIP prefixes -> one row per ZIP
# median so a few bad points don't pull the location off
geolocation_by_zip = geo.groupby("geolocation_zip_code_prefix").agg({
    "geolocation_lat": "median",
    "geolocation_lng": "median",
    "geolocation_city": "first",
    "geolocation_state": "first",
}).reset_index()

print("before:", len(data["geolocation"]))
print("after:", len(geolocation_by_zip))

## 9. Checks and save

In [ ]:
# one row per key in each of these, all should be True
print("orders:", data["orders"]["order_id"].is_unique)
print("payments_by_order:", payments_by_order["order_id"].is_unique)
print("reviews_by_order:", reviews_by_order["order_id"].is_unique)
print("geolocation_by_zip:", geolocation_by_zip["geolocation_zip_code_prefix"].is_unique)
print("orders row count still 99441:", len(data["orders"]) == 99441)

In [ ]:
data["customers"].to_csv(PROCESSED_PATH + "customers_clean.csv", index=False)
data["orders"].to_csv(PROCESSED_PATH + "orders_clean.csv", index=False)
data["order_items"].to_csv(PROCESSED_PATH + "order_items_clean.csv", index=False)
data["products"].to_csv(PROCESSED_PATH + "products_clean.csv", index=False)
data["sellers"].to_csv(PROCESSED_PATH + "sellers_clean.csv", index=False)
data["payments"].to_csv(PROCESSED_PATH + "payments_clean.csv", index=False)
payments_by_order.to_csv(PROCESSED_PATH + "payments_by_order_clean.csv", index=False)
reviews_by_order.to_csv(PROCESSED_PATH + "reviews_by_order_clean.csv", index=False)
geolocation_by_zip.to_csv(PROCESSED_PATH + "geolocation_by_zip_clean.csv", index=False)

print("saved")

## Summary

- converted dates, fixed ZIP code zeros
- products: fixed column names, 610 labelled `unknown`, all categories translated
- orders: flagged 8 delivered-without-date and 6 canceled-with-delivery
- 96,470 orders in analysis scope
- added `customer_unique_id`, `purchase_month`, `delivery_days`, `is_late`
- revenue = price + freight per item
- payments -> one row per order (3 `not_defined` payments are all 0)
- reviews 99,224 -> 98,673 (latest per order)
- geolocation 1,000,163 -> 19,010 ZIP prefixes